Packages Initial

In [2]:
import pandas as pd
import numpy as np
import shutil
import openpyxl
import os
#R
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

R pathway temp

In [2]:
r_home = r'C:\Program Files\R\R-4.5.3'
os.environ['R_HOME'] = r_home

# Add the R 'bin\x64' folder to the Windows system PATH so Python can see the .dll files
r_bin = os.path.join(r_home, 'bin', 'x64')
os.environ['PATH'] = r_bin + ';' + os.environ.get('PATH', '')

---

### Template Maker

---


Initializer

In [3]:
def initialize_project_file(first_base_name, new_filename, master_template_path="Template.xlsx"):
    """
    Creates the main project file, generates the first set of sheets,
    and hides the master sheets so they don't clutter the workspace.
    """
    # 1. Copy the master template
    shutil.copy(master_template_path, new_filename)
    wb = openpyxl.load_workbook(new_filename)

    # 2. Duplicate the first set (e.g., "Glass_9um")
    wb.copy_worksheet(wb['工作表1']).title = f"{first_base_name}"
    wb.copy_worksheet(wb['工作表2']).title = f"{first_base_name}_data"
    wb.copy_worksheet(wb['工作表3']).title = f"{first_base_name}_Result"

    # 3. THE TRICK: Hide the original sheets instead of deleting them
    for sheet_name in ['工作表1', '工作表2', '工作表3']:
        wb[sheet_name].sheet_state = 'hidden'

    wb.save(new_filename)
    print(f"✅ Project initialized: '{new_filename}' with '{first_base_name}'")


Template testing

In [4]:
def add_set_to_existing_file(new_base_name, existing_filename):
    """
    Opens an existing project file, copies the hidden template sheets,
    names them, and saves the file.
    """
    # 1. Open the file you already created
    try:
        wb = openpyxl.load_workbook(existing_filename)
    except FileNotFoundError:
        print(f"Error: '{existing_filename}' does not exist yet.")
        return

    # 2. Duplicate from the hidden master sheets
    sheet_x = wb.copy_worksheet(wb['工作表1'])
    sheet_x.title = f"{new_base_name}"

    sheet_x_data = wb.copy_worksheet(wb['工作表2'])
    sheet_x_data.title = f"{new_base_name}_data"

    sheet_x_result = wb.copy_worksheet(wb['工作表3'])
    sheet_x_result.title = f"{new_base_name}_Result"

    # 3. Save the updated file
    wb.save(existing_filename)
    print(f"✅ Added new set: '{new_base_name}' to '{existing_filename}'")

In [5]:
print("=== DRG Analysis Template Generator ===")
print("1: Start a NEW project file (Creates file and first set of sheets)")
print("2: ADD a new set of sheets to an EXISTING project file")
choice = input("Enter 1 to create new file or 2 to add sets: ")

target_filename = "Results.xlsx"

if choice == '1':
    # Ask for the custom name
    custom_name = input("Enter the condition name for your FIRST set (e.g., Glass_9um): ")

    # Safety check so you don't accidentally overwrite hours of work
    if os.path.exists(target_filename):
        overwrite = input(f"WARNING: '{target_filename}' already exists. Overwrite it? (y/n): ")
        if overwrite.lower() != 'y':
            print("Operation cancelled.")
            exit()

    initialize_project_file(custom_name, target_filename)

elif choice == '2':
    # Ask for the custom name
    custom_name = input("Enter the condition name for your NEW set (e.g., Quartz_39um): ")
    add_set_to_existing_file(custom_name, target_filename)

else:
    print("Invalid choice. Please run the script again.")

=== DRG Analysis Template Generator ===
1: Start a NEW project file (Creates file and first set of sheets)
2: ADD a new set of sheets to an EXISTING project file
✅ Project initialized: 'Results.xlsx' with 'test_1'


---
Template Droplist FIX

---

In [ ]:
#def

---
region.RGN  to .CSV / MetaFlour

In [2]:
def load_metafluor_rgn(file_path):
    regions = []
    with open(file_path, 'r') as f:
        for line in f:
            # RGN lines are comma-separated pairs of "Tag Value"
            parts = line.strip().split(',')
            row_dict = {}
            for p in parts:
                item = p.strip().split(' ')
                if len(item) >= 2:
                    tag = item[0]
                    val = " ".join(item[1:])
                    row_dict[tag] = val
            regions.append(row_dict)

    df = pd.DataFrame(regions)
    # Tag '2' typically contains "X Y" coordinates of the top-left corner
    coords = df['2'].str.split(' ', expand=True)
    x_raw = pd.to_numeric(coords[0])
    y_raw = pd.to_numeric(coords[1].str.replace(',', '')) # Handle potential commas

    # Apply your specific scaling and Y-inversion from the MATLAB script
    df['x'] = 0.7752 * x_raw
    df['y'] = 0.7752 * (512 - y_raw)
    return df

Original MATLAB calcium data processor_Python ver.

code testing

In [ ]:
# 1. Load Intensity Data
# MATLAB: original_data = readmatrix('import_data.xlsx')
original_data = pd.read_excel('import_data.xlsx', header=None).values

# 2. Extract 340 and 380 Intensities
# Every 3rd column is skipped; then alternated between 340 and 380
intensity = original_data[:, [i for i in range(original_data.shape[1]) if i % 3 != 0]]
intensity_340 = intensity[:, 0::2] - intensity[:, [0]]
intensity_380 = intensity[:, 1::2] - intensity[:, [1]]

# 3. Calculate Ratios and F0/Fp
with np.errstate(divide='ignore', invalid='ignore'):
    Ratio = np.where(intensity_380 != 0, intensity_340 / intensity_380, 0)
W = Ratio.shape[1]

# Input stimulation time (e.g., 60)
us_start = int(input('Please input the beginning time for US stimulation (s): '))

# F0: Mean of 5 seconds before US
f0 = np.mean(Ratio[us_start-6 : us_start-1, :], axis=0)
# Fp: Max of 30 seconds after US
fp_window = Ratio[us_start : us_start+30, :]
fp = np.max(fp_window, axis=0)

# Suppress the zero-division warning for the background region
with np.errstate(divide='ignore', invalid='ignore'):
    # If f0 is 0, set result to 0; otherwise, perform the calculation
    df_f0 = np.where(f0 != 0, (fp - f0) / f0, 0)

# 4. Read Region Data Directly (RGN modification)
rgn_df = load_metafluor_rgn('region.RGN')
x = rgn_df['x'].values
y = rgn_df['y'].values

# 5. Geometric Analysis (Displacement & Angle)
# Reference points from region #2 and #3
x2, y2 = x[1], y[1]
x3, y3 = x[2], y[2]

displacement = np.sqrt((x - x2)**2 + (y - y2)**2)

# Vector calculations
v_line = np.array([x2 - x3, y2 - y3])
v_cells = np.stack([x - x2, y - y2], axis=1)

dot_product = np.dot(v_cells, v_line)
norms = np.linalg.norm(v_line) * np.linalg.norm(v_cells, axis=1)
with np.errstate(divide='ignore', invalid='ignore'):
    angle_temp = np.degrees(np.arccos(np.clip(dot_product / norms, -1.0, 1.0)))

# Angle correction logic from MATLAB
final_angles = []
for i in range(len(x)):
    slope_cell = (y[i] - y3) / (x[i] - x3) if (x[i] - x3) != 0 else 0
    slope_ref = v_line[1] / v_line[0] if v_line[0] != 0 else 0

    if slope_cell > slope_ref or slope_cell > 0:
        final_angles.append(360 - angle_temp[i])
    else:
        final_angles.append(angle_temp[i])

# 6. Output Generation
i_us = intensity_340[us_start + 2, :]

output_data = pd.DataFrame({
    'Number': np.arange(1, W + 1),
    'dF/F0': df_f0,
    '340/380(True=-)': (intensity_340[us_start, :] - i_us) * (intensity_380[us_start, :] - (intensity_380[us_start+2, :])), # q_ratio logic
    'Intensity_340(US+3sec)': i_us,
    'Displacement': displacement,
    'Angle': final_angles
})


output_data.to_excel('Data_Output.xlsx', index=False)

print("Processing complete. Files saved.")

Python Processor / MetaFlour

In [9]:
def process_calcium_data():
    print("\n--- Processing Calcium Imaging Data ---")

    # 1. Load Intensity Data
    try:
        original_data = pd.read_excel('import_data.xlsx', header=None).values
    except FileNotFoundError:
        print("❌ Error: 'import_data.xlsx' not found.")
        return

    # 2. Extract 340 and 380 Intensities
    intensity = original_data[:, [i for i in range(original_data.shape[1]) if i % 3 != 0]]
    intensity_340 = intensity[:, 0::2] - intensity[:, [0]]
    intensity_380 = intensity[:, 1::2] - intensity[:, [1]]

    # 3. Calculate Global Ratios
    with np.errstate(divide='ignore', invalid='ignore'):
        Ratio = np.where(intensity_380 != 0, intensity_340 / intensity_380, 0)
    W = Ratio.shape[1]

    # 4. Read Region Data Directly (Ensure your custom function is defined in your environment)
    try:
        # NOTE: Make sure your load_metafluor_rgn function is defined above this or imported!
        rgn_df = load_metafluor_rgn('region.RGN')
        x, y = rgn_df['x'].values, rgn_df['y'].values
    except Exception as e:
        print(f"❌ Error loading RGN file: {e}")
        return

    # 5. Geometric Analysis (Calculated once)
    x2, y2 = x[1], y[1]
    x3, y3 = x[2], y[2]
    displacement = np.sqrt((x - x2)**2 + (y - y2)**2)

    v_line = np.array([x2 - x3, y2 - y3])
    v_cells = np.stack([x - x2, y - y2], axis=1)
    dot_product = np.dot(v_cells, v_line)
    norms = np.linalg.norm(v_line) * np.linalg.norm(v_cells, axis=1)

    with np.errstate(divide='ignore', invalid='ignore'):
        angle_temp = np.degrees(np.arccos(np.clip(dot_product / norms, -1.0, 1.0)))

    final_angles = []
    for i in range(len(x)):
        slope_cell = (y[i] - y3) / (x[i] - x3) if (x[i] - x3) != 0 else 0
        slope_ref = v_line[1] / v_line[0] if v_line[0] != 0 else 0
        if slope_cell > slope_ref or slope_cell > 0:
            final_angles.append(360 - angle_temp[i])
        else:
            final_angles.append(angle_temp[i])

    # 6. Handle Jupyter-Friendly Input (One popup, space-separated)
    print("\nEnter beginning times for US stimulation.")
    raw_input = input("Type your times separated by spaces (e.g., 60 180 300) and press Enter: ")

    # This magic line replaces any accidental commas with spaces,
    # splits the numbers up, and converts them into a list of integers all at once!
    us_start_list = [int(x) for x in raw_input.replace(',', ' ').split() if x.strip()]

    if not us_start_list:
        print("No time points entered. Cancelling calculation.")
        return

    # 7. Loop through times and save to multi-sheet Excel
    output_filename = 'Data_Output_Multiple.xlsx'
    with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
        for us_start in us_start_list:
            # Mathematics for this specific time point
            f0 = np.mean(Ratio[us_start-6 : us_start-1, :], axis=0)
            fp_window = Ratio[us_start : us_start+30, :]
            fp = np.max(fp_window, axis=0)

            with np.errstate(divide='ignore', invalid='ignore'):
                df_f0 = np.where(f0 != 0, (fp - f0) / f0, 0)

            i_us = intensity_340[us_start + 2, :]
            q_ratio = (intensity_340[us_start, :] - i_us) * (intensity_380[us_start, :] - intensity_380[us_start+2, :])

            # Bundle data
            output_data = pd.DataFrame({
                'Number': np.arange(1, W + 1),
                'dF/F0': df_f0,
                '340/380(True=-)': q_ratio,
                'Intensity_340(US+3sec)': i_us,
                'Displacement': displacement,
                'Angle': final_angles
            })

            sheet_name = f"Result_{us_start}s"
            output_data.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  -> Calculated and saved: {sheet_name}")

    print(f"✅ All time points processed! Saved to '{output_filename}'\n")

Simple processor

In [3]:
def simple_calcium_process():
    print("\n--- Processing Simplified Calcium Imaging Data ---")

    # 1. Load Intensity Data
    try:
        original_data = pd.read_excel('import_data.xlsx', header=None).values
    except FileNotFoundError:
        print("❌ Error: 'import_data.xlsx' not found.")
        return

    # 2. Extract 340 and 380 Intensities
    intensity = original_data[:, [i for i in range(original_data.shape[1]) if i % 3 != 0]]
    intensity_340 = intensity[:, 0::2] - intensity[:, [0]]
    intensity_380 = intensity[:, 1::2] - intensity[:, [1]]

    # 3. Calculate Global Ratios
    with np.errstate(divide='ignore', invalid='ignore'):
        Ratio = np.where(intensity_380 != 0, intensity_340 / intensity_380, 0)
    W = Ratio.shape[1]

    # 4. Handle Jupyter-Friendly Input
    print("\nEnter beginning times for US stimulation.")
    raw_input = input("Type your times separated by spaces (e.g., 60 180 300) and press Enter: ")

    us_start_list = [int(x) for x in raw_input.replace(',', ' ').split() if x.strip()]

    if not us_start_list:
        print("No time points entered. Cancelling calculation.")
        return

    # 5. Initialize the output DataFrame with the ROI/Cell numbers
    output_data = pd.DataFrame({'Number': np.arange(1, W + 1)})

    # 6. Loop through times, calculate dF/F0, and add as new columns
    for us_start in us_start_list:
        f0 = np.mean(Ratio[us_start-6 : us_start-1, :], axis=0)
        fp_window = Ratio[us_start : us_start+30, :]
        fp = np.max(fp_window, axis=0)

        with np.errstate(divide='ignore', invalid='ignore'):
            df_f0 = np.where(f0 != 0, (fp - f0) / f0, 0)

        # Append the new column to the DataFrame
        col_name = f"dF/F0_{us_start}s"
        output_data[col_name] = df_f0
        print(f"  -> Calculated: {col_name}")

    # 7. Read Region Data Directly (Ensure your custom function is defined in your environment)
    try:
        rgn_df = load_metafluor_rgn('region.RGN')
        x, y = rgn_df['x'].values, rgn_df['y'].values
    except Exception as e:
        print(f"❌ Error loading RGN file: {e}")
        return

    # 8. Geometric Analysis (Calculated once)
    x2, y2 = x[1], y[1]
    x3, y3 = x[2], y[2]
    displacement = np.sqrt((x - x2)**2 + (y - y2)**2)

    v_line = np.array([x2 - x3, y2 - y3])
    v_cells = np.stack([x - x2, y - y2], axis=1)
    dot_product = np.dot(v_cells, v_line)
    norms = np.linalg.norm(v_line) * np.linalg.norm(v_cells, axis=1)

    with np.errstate(divide='ignore', invalid='ignore'):
        angle_temp = np.degrees(np.arccos(np.clip(dot_product / norms, -1.0, 1.0)))

    final_angles = []
    for i in range(len(x)):
        slope_cell = (y[i] - y3) / (x[i] - x3) if (x[i] - x3) != 0 else 0
        slope_ref = v_line[1] / v_line[0] if v_line[0] != 0 else 0
        if slope_cell > slope_ref or slope_cell > 0:
            final_angles.append(360 - angle_temp[i])
        else:
            final_angles.append(angle_temp[i])

    output_data['Displacement'] = displacement
    output_data['Angle'] = final_angles

    # 9. Save to a single Excel sheet
    output_filename = 'Data_Output_Simplified.xlsx'
    output_data.to_excel(output_filename, index=False)
    print(f"✅ All time points processed! Saved to '{output_filename}'\n")


In [20]:
simple_calcium_process()


--- Processing Simplified Calcium Imaging Data ---

Enter beginning times for US stimulation.
  -> Calculated: dF/F0_60s
  -> Calculated: dF/F0_180s
  -> Calculated: dF/F0_300s
  -> Calculated: dF/F0_420s
✅ All time points processed! Saved to 'Data_Output_Simplified.xlsx'



---

Zens Processor /No displacement & angle calculation

*ＦＯＲ ＦＵＴＵＲＥ　ＵＰＤＡＴＥＳ*

---

In [8]:
def new_processor():
    # --- 1. Load Data ---
    # Column A (index 0) is background, Column B (index 1) starts the data
    try:
        original_data = pd.read_excel('new_import.xlsx', header=None).values
        # MATLAB: Ratio = original_data(:, 2:end);[cite: 4]
        ratio_data = original_data[:, 1:]
        L, W = ratio_data.shape
    except FileNotFoundError:
        print(" Error: 'new_import.xlsx' not found.")
        return

    # --- 2. Handle Multiple Inputs (Jupyter-Friendly) ---
    print(f"Total frames detected: {L}")
    print("Enter stimulation start times (e.g., 60 180 300):")
    raw_input = input("Times separated by spaces: ")

    # Clean input: replaces commas with spaces and converts to integer list
    us_times = [int(t) for t in raw_input.replace(',', ' ').split() if t.strip()]

    if not us_times:
        print("No timepoints entered. Stopping.")
        return

    # Dictionary to store dataframes for Excel export
    all_results = {}

    for us in us_times:
        print(f"\nProcessing US stimulation at {us}s...")

        # 1. Baseline (The "Excel AVG" of rows US-5 to US-1)
        #  us-6 : for 55-59s
        baseline_range = ratio_data[us-6 : us-1, :]
        f0 = np.mean(baseline_range, axis=0)

        # 2. Peak (MAX US+1 to US+30)
        end_idx = min(us+30, L)
        peak_range = ratio_data[us : end_idx, :]
        fp = np.max(peak_range, axis=0)

        # 3. dF/F0 Calculation
        # bypass ZeroDivisionError with np.divide : if f0 is 0, put NaN"
        numerator = fp - f0
        df_f0 = np.divide(numerator, f0,
                          out=np.full_like(f0, np.nan),
                          where=f0 != 0)

        # 4. Bundle into the table
        df_output = pd.DataFrame({
            'Number': np.arange(1, W + 1),
            'dF/F0': df_f0
        })

        all_results[f"US_{us}s"] = df_output

    # --- 4. Data Output ---
    # Saves each stimulation time to a different sheet in the same file
    output_filename = 'New_Data_Output_Multiple.xlsx'
    with pd.ExcelWriter(output_filename) as writer:
        for sheet_name, data in all_results.items():
            data.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"\n✅ All results saved to '{output_filename}'")

__Error Diagnosis__

In [ ]:
import pandas as pd
import numpy as np

def process_calcium_diagnostic():
    try:
        # Load data with float64 precision
        original_data = pd.read_excel('new_import.xlsx', header=None, dtype=np.float64).values
        ratio_data = original_data[:, 1:]
        L, W = ratio_data.shape
    except FileNotFoundError:
        print("❌ Error: 'new_import.xlsx' not found.")
        return

    print("Enter stimulation times (e.g., 60 180):")
    raw_input = input("Times: ")
    us_times = [int(t) for t in raw_input.replace(',', ' ').split() if t.strip()]

    all_results = {}

    for us in us_times:
        # 1. Calculate Exact Baseline (F0)
        baseline_range = ratio_data[us-5 : us, :]
        f0 = np.mean(baseline_range, axis=0)

        # 2. Calculate Exact Peak (Fp)
        end_idx = min(us+30, L)
        peak_range = ratio_data[us+1 : end_idx, :]
        fp = np.max(peak_range, axis=0)

        # 3. Safe Division for dF/F0
        numerator = fp - f0
        df_f0 = np.divide(numerator, f0,
                          out=np.full_like(f0, np.nan),
                          where=f0 != 0)

        # 4. DIAGNOSTIC STORAGE: Save F0, Fp, and dF/F0 together
        df_output = pd.DataFrame({
            'Number': np.arange(1, W + 1),
            'F0 (Baseline)': f0,
            'Fp (Peak)': fp,
            'dF/F0': df_f0
        })

        all_results[f"US_{us}s"] = df_output

    # 5. Excel Output
    output_filename = 'Diagnostic_Data_Output.xlsx'
    with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
        for sheet_name, data in all_results.items():
            data.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"\n✅ Diagnostic file saved to '{output_filename}'")

process_calcium_diagnostic()

Enter stimulation times (e.g., 60 180):


Run the function

In [ ]:
new_processor()

Total frames detected: 484
Enter stimulation start times (e.g., 60 180 300):


## Test Run

In [9]:
if __name__ == "__main__":
    while True:
        print("=== DRG Analysis & Statistics Workspace ===")
        print("1: Process Raw Data & Calculate dF/F0 (Multiple Times)")
        print("2: Start a NEW Statistical Project File")
        print("3: ADD a Condition to an EXISTING Project File")
        print("4: Exit")

        choice = input("Select an option (1-4): ")

        if choice == '1':
            process_calcium_data()

        elif choice == '2':
            target_filename = input("Name your new statistical project file (e.g., April_Run): ")
            if not target_filename.endswith('.xlsx'): target_filename += '.xlsx'

            if os.path.exists(target_filename):
                if input(f"'{target_filename}' exists. Overwrite? (y/n): ").lower() != 'y': continue

            custom_name = input("Enter the condition name for your FIRST set (e.g., Glass_9um): ")
            initialize_project_file(custom_name, target_filename)

        elif choice == '3':
            target_filename = input("Which existing project file do you want to open?: ")
            if not target_filename.endswith('.xlsx'): target_filename += '.xlsx'

            custom_name = input("Enter the condition name for your NEW set (e.g., Quartz_39um): ")
            add_set_to_existing_file(custom_name, target_filename)

        elif choice == '4':
            print("Exiting workspace. Good luck with the research!")
            break
        else:
            print("Invalid choice. Try again.\n")

=== DRG Analysis & Statistics Workspace ===
1: Process Raw Data & Calculate dF/F0 (Multiple Times)
2: Start a NEW Statistical Project File
3: ADD a Condition to an EXISTING Project File
4: Exit

--- Processing Calcium Imaging Data ---

Enter beginning times for US stimulation.
  -> Calculated and saved: Result_60s
  -> Calculated and saved: Result_180s
  -> Calculated and saved: Result_300s
  -> Calculated and saved: Result_420s
✅ All time points processed! Saved to 'Data_Output_Multiple.xlsx'

=== DRG Analysis & Statistics Workspace ===
1: Process Raw Data & Calculate dF/F0 (Multiple Times)
2: Start a NEW Statistical Project File
3: ADD a Condition to an EXISTING Project File
4: Exit
Exiting workspace. Good luck with the research!


---
R Analysis

In [14]:
r_script = """
    library(stats)
    library(ARTool)
    library(emmeans)

    # 1. Prepare Factors
    r_data$Substrate <- as.factor(r_data$Substrate)
    r_data$US_Parameter <- as.factor(r_data$US_Parameter)

    # 2. Run ART Model
    m <- art(Value ~ Substrate * US_Parameter, data = r_data)

    aov_results <- as.data.frame(anova(m))
        # 為 ANOVA 加上星號 (根據 Pr(>F) 欄位)
    aov_results$sig <- symnum(aov_results$`Pr(>F)`,
                              corr = FALSE, na = FALSE,
                              cutpoints = c(0, 0.0001, 0.001, 0.01, 0.05, 0.1, 1),
                              symbols = c("****","***", "**", "*", ".", " "))

    #| label: 事後分析
    # 1. 取得原始、未校正的 28 組數據
    res_raw <- art.con(m, "Substrate:US_Parameter", adjust = "none")
    res_df <- as.data.frame(res_raw)

    # 2. 定義篩選邏輯
    splits <- strsplit(as.character(res_df$contrast), " - ")

    # A. 找出「同參數下比材質」的 4 組
    is_substrate_comp <- sapply(splits, function(x) {
      p_left  <- sub("^[^,]+,", "", x[1])
      p_right <- sub("^[^,]+,", "", x[2])
      return(p_left == p_right)
    })

    # B. 找出「同材質下比參數」的 12 組
    is_parameter_comp <- sapply(splits, function(x) {
      s_left  <- sub(",.*$", "", x[1])
      s_right <- sub(",.*$", "", x[2])
      return(s_left == s_right)
    })

    # 3. MOD
    sub_sme <- res_df[is_substrate_comp, ]
    sub_sme$p.adj <- p.adjust(sub_sme$p.value, method = "fdr")

    param_sme <- res_df[is_parameter_comp, ]
    param_sme$p.adj <- p.adjust(param_sme$p.value, method = "fdr")

    final_results <- rbind(
      data.frame(sub_sme, Type = "Substrate_Effect"),
      data.frame(param_sme, Type = "Parameter_Effect")
    )

    final_results$sig <- cut(final_results$p.adj,
                             breaks=c(-Inf, 0.001, 0.01, 0.05, 0.1, 1),
                             labels=c("***", "**", "*", ".", "ns"))

    # 5. RETURN THE LIST (No write_xlsx needed!)
    list(anova_table = aov_results, posthoc_table = final_results)
    """

ART ANOVA

In [15]:
def run_art_anova(excel_filepath, sheet_name):
    print("\n--- Running ART-ANOVA via PETRA ---")

    # 1. Read the data
    df = pd.read_excel(excel_filepath, sheet_name=sheet_name)

    # R hates slashes in variable names, so we clean it up before sending
    if 'dF/F0' in df.columns:
        df.rename(columns={'dF/F0': 'dF_F0'}, inplace=True)

    print("Calculating statistics in R background...")

    with localconverter(robjects.default_converter + pandas2ri.converter):
        # Inject the Pandas dataframe into R environment
        robjects.globalenv['r_data'] = robjects.conversion.py2rpy(df)

        # Execute the R string
        r_result = robjects.r(r_script)

        # Convert the R results back into Pandas DataFrames
        anova_df = robjects.conversion.rpy2py(r_result[0])
        posthoc_df = robjects.conversion.rpy2py(r_result[1])

    print("✅ Statistics complete! DataFrames ready for Excel.")

    return anova_df, posthoc_df

Test ART ANOVA

In [ ]:
if __name__ == "__main__":
    try:
        anova_results, posthoc_results = run_art_anova("data.xlsx", "工作表1")

        print("\n--- Python sees the ANOVA Table: ---")
        print(anova_results)

        print("\n--- Python sees the Post-Hoc Table: ---")
        print(posthoc_results)
    except Exception as e:
        print(f"\n❌ Test failed: {e}")


數據格式轉換



In [8]:
file_path = "Results.xlsx"

try:
    # Load the workbook to get list of sheet names
    xl = pd.ExcelFile(file_path)
    sheet_names = xl.sheet_names

    # Open the Excel file in append mode ('a') to modify it without deleting untouched sheets.
    # if_sheet_exists='replace' lets us overwrite the specific output sheets.
    with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:

        # Loop through indices with a step of 3 (0, 3, 6... which are the 1st, 4th, 7th sheets)
        for i in range(0, len(sheet_names), 3):
            src_idx = i         # Source sheet index
            tgt_idx = i + 1     # Target sheet index

            # Break if we've reached the end of the sheets
            if tgt_idx >= len(sheet_names):
                break

            src_sheet = sheet_names[src_idx]
            tgt_sheet = sheet_names[tgt_idx]

            print(f" Reading from '{src_sheet}' -> Writing to '{tgt_sheet}'...")

            # Read the specific source sheet
            df_raw = pd.read_excel(file_path, sheet_name=src_sheet, header=None)

            results = []
            detected_substrates = set()
            detected_params = set()

            # Iterate through columns starting from column 1
            for col in range(1, df_raw.shape[1]):
                # Substrate (Row 1)
                sub = df_raw.iloc[1, col]
                if pd.isna(sub) or str(sub).strip().lower() == 'substrate':
                    continue

                substrate_name = str(sub).strip()
                detected_substrates.add(substrate_name)

                # Parameter (Row 2)
                param = df_raw.iloc[2, col]
                if pd.isna(param) or str(param).strip().lower() == 'us_parameter':
                    continue

                param_name = str(param).strip()
                detected_params.add(param_name)

                # Values (Row 3 onwards)
                vals = pd.to_numeric(df_raw.iloc[3:, col], errors='coerce').dropna()
                for v in vals:
                    results.append([param_name, v, substrate_name])

            # If we found data, create the DataFrame and write it to the target sheet
            if results:
                final_df = pd.DataFrame(results, columns=['US_Parameter', 'Value', 'Substrate'])
                final_df.to_excel(writer, sheet_name=tgt_sheet, index=False)
                print(f"   Success! Updated '{tgt_sheet}' with {len(final_df)} rows.")
            else:
                print(f"   No valid data found in '{src_sheet}'. Skipping.")

    print("\n Done")

except Exception as e:
    print(f" Error occurred: {e}")

 Reading from '工作表1' -> Writing to '工作表2'...
   No valid data found in '工作表1'. Skipping.
 Reading from 'TV-KO_Glass_高度比較' -> Writing to 'TV-KO_Glass_高度比較_data'...
   Success! Updated 'TV-KO_Glass_高度比較_data' with 446 rows.
 Reading from 'TV-KO_Quartz_高度比較' -> Writing to 'TV-KO_Quartz_高度比較_data'...
   Success! Updated 'TV-KO_Quartz_高度比較_data' with 405 rows.
 Reading from 'TV-KO_39um_底材比較' -> Writing to 'TV-KO_39um_底材比較_data'...
   Success! Updated 'TV-KO_39um_底材比較_data' with 453 rows.
 Reading from 'TV-KO_9um_底材比較' -> Writing to 'TV-KO_9um_底材比較_data'...
   Success! Updated 'TV-KO_9um_底材比較_data' with 398 rows.

 Done
